# COMAR RAG — Complete Ingestion Pipeline
### Google Colab Edition (T4 GPU recommended)

This notebook downloads all COMAR (Code of Maryland Regulations) XML files from
GitHub, parses them into regulation dicts, splits them into embeddable chunks,
builds a knowledge graph, and uploads everything to **Qdrant Cloud** using
BGE-M3 dense + sparse vectors.

| Step | What happens | Approx. time |
|------|-------------|------|
| 1 | Install packages | 3 min |
| 2 | Mount Google Drive | 30 s |
| 3 | Download ~422 XML files from GitHub | 10–15 min |
| 4 | Parse 3,309 regulations from XML | 2 min |
| 5 | Create ~52,528 chunks | 1 min |
| 6 | Build NetworkX knowledge graph | 1 min |
| 7 | Set up Qdrant Cloud collection | 1 min |
| 8 | Load BGE-M3 (downloads 2.3 GB weights) | 5 min |
| 9 | Embed + upload all chunks | 2–3 hours |
| 10 | Verify & test search | 1 min |
| 11 | Create snapshot for local transfer | 5 min |

**Before starting, you need:**
- A **Qdrant Cloud** account (free at https://cloud.qdrant.io) — create a cluster and copy the URL and API key.
- (Optional) A **GitHub personal access token** — get one at https://github.com/settings/tokens (no scopes needed). Without it you get 60 API requests/hour which may hit the rate limit.

---

> **Tip**: Enable GPU first! Go to **Runtime → Change runtime type → T4 GPU**.
> Without a GPU, embedding will take ~40 hours instead of ~3 hours.

In [ ]:
# ============================================================
# STEP 0: Check GPU
# ============================================================
import subprocess, sys, platform

print('=== Runtime Info ===')
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'\n✅ GPU detected:')
    for line in result.stdout.strip().splitlines():
        print(f'   {line}')
else:
    print('\n⚠️  No GPU detected!')
    print('   Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   Embedding without GPU will take ~40x longer.')

In [ ]:
# ============================================================
# STEP 1: Install Dependencies
# ============================================================
print('Installing packages...')
!pip install -q lxml requests tqdm tiktoken networkx qdrant-client
!pip install -q "FlagEmbedding>=1.3,<2.0"
print('\n✅ All packages installed.')

# Quick sanity check
import importlib
for pkg in ['lxml', 'tqdm', 'tiktoken', 'networkx', 'qdrant_client', 'FlagEmbedding']:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', '?')
        print(f'  {pkg}: {ver}')
    except ImportError:
        print(f'  {pkg}: MISSING')


In [ ]:
# ============================================================
# STEP 2: Mount Google Drive
# ============================================================
# Google Drive persists data between Colab sessions.
# Even if your runtime disconnects and restarts, the XML files
# and upload checkpoint will still be there.
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# All data goes under this Drive folder
BASE_DIR   = '/content/drive/MyDrive/COMAR_RAG'
DATA_DIR   = f'{BASE_DIR}/xml_cache'      # downloaded XML files
GRAPH_DIR  = f'{BASE_DIR}/data'           # graph, definitions, regulations JSON
CKPT_FILE  = f'{BASE_DIR}/upload_checkpoint.txt'  # resume token for uploads

for d in [DATA_DIR, GRAPH_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'\n✅ Google Drive mounted.')
print(f'   Base directory : {BASE_DIR}')
print(f'   XML cache      : {DATA_DIR}')
print(f'   Processed data : {GRAPH_DIR}')

In [ ]:
# ============================================================
# STEP 3: Configuration  <-- EDIT THIS CELL
# ============================================================

# --- GitHub (optional but recommended) ----------------------
# Without a token you get 60 API requests/hour (may hit limit).
# With a token: 5,000 requests/hour. No special scopes needed.
# Get one at: https://github.com/settings/tokens
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxxxxxxxxxx'

# --- Qdrant Cloud -------------------------------------------
# 1. Sign up free at https://cloud.qdrant.io
# 2. Click "Create Cluster" -> choose any region -> Free tier
# 3. After cluster is ready, copy:
#    - Endpoint URL  (looks like https://abc-123.us-east4-0.gcp.cloud.qdrant.io)
#    - API Key       (a long alphanumeric token)
QDRANT_URL     = ''   # paste your cluster endpoint URL here
QDRANT_API_KEY = ''   # paste your API key here

# --- Collection name (leave as-is to match local project) ---
COLLECTION_NAME = 'comar_regulations'

# --- Titles to ingest ---------------------------------------
# Title 15 = Agriculture (~1,093 regulations)
# Title 26 = Environment (~2,216 regulations)
TITLES = ['15', '26']

# --- Embedding batch size -----------------------------------
# Reduce to 8 if you get CUDA out-of-memory errors.
EMBED_BATCH_SIZE = 16

# ---- Validation --------------------------------------------
print('✅ Configuration loaded.')
if GITHUB_TOKEN:
    print(f'   GitHub token   : set (will use authenticated API)')
else:
    print('   GitHub token   : NOT set (unauthenticated, 60 req/hr limit)')

if QDRANT_URL and QDRANT_API_KEY:
    print(f'   Qdrant Cloud   : {QDRANT_URL}')
else:
    print('   Qdrant Cloud   : ⚠️  NOT configured -- set QDRANT_URL and QDRANT_API_KEY above!')

---
## Step 1: Download COMAR XML Files from GitHub

COMAR regulations are published as open XML files in the
[`maryland-dsd/law-xml`](https://github.com/maryland-dsd/law-xml) repository.

The next two cells:
1. Use the GitHub Trees API to discover all XML file paths under Title 15 and Title 26.
2. Download each file to Google Drive (cached — re-runs skip already-downloaded files).

**Expected files**: ~162 for Title 15, ~260 for Title 26 (total ~422 XML files).

In [ ]:
# ============================================================
# Define fetch helpers (no ML models needed here)
# ============================================================
import os, time, requests
from pathlib import Path
from tqdm import tqdm

GITHUB_API  = 'https://api.github.com'
REPO        = 'maryland-dsd/law-xml'
BRANCH      = 'main'
COMAR_BASE  = 'us/md/exec/comar'

# Build authenticated session
_session = requests.Session()
_headers = {'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
if GITHUB_TOKEN:
    _headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'
_session.headers.update(_headers)


def _github_get(url, params=None):
    """GET a GitHub API URL with rate-limit retry."""
    for attempt in range(3):
        r = _session.get(url, params=params, timeout=60)
        if r.status_code == 403 and 'rate limit' in r.text.lower():
            reset_ts = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            wait = max(reset_ts - int(time.time()), 10)
            print(f'  Rate limit hit — waiting {wait}s ...')
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f'GitHub API failed after 3 retries: {url}')


def list_title_blobs(title):
    """Return all XML repo-paths for a given COMAR title."""
    data = _github_get(
        f'{GITHUB_API}/repos/{REPO}/git/trees/{BRANCH}',
        params={'recursive': '1'}
    )
    if data.get('truncated'):
        print('  ⚠️  Tree response was truncated. Set GITHUB_TOKEN to avoid this.')
    prefix = f'{COMAR_BASE}/{title}/'
    return [
        item['path']
        for item in data.get('tree', [])
        if item['type'] == 'blob'
        and item['path'].startswith(prefix)
        and item['path'].endswith('.xml')
    ]


def download_file(repo_path, dest, refresh=False):
    """Download a single raw file from GitHub.

    Retries up to 3 times to handle Google Drive FUSE latency:
    Drive sometimes needs a moment after mkdir before a file can be written.
    """
    dest = Path(dest)
    if dest.exists() and not refresh:
        return  # cache hit

    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{repo_path}'
    r = _session.get(url, timeout=60)
    r.raise_for_status()
    content = r.content

    for attempt in range(3):
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(content)
            return  # success
        except FileNotFoundError:
            if attempt < 2:
                time.sleep(0.5)   # give Drive FUSE time to register the directory
            else:
                raise


print('✅ Fetch helpers defined.')


In [ ]:
# ============================================================
# Discover and download all XML files
# ============================================================
all_blobs = {}  # title -> list of repo-relative paths

for title in [t.zfill(2) for t in TITLES]:
    print(f'Fetching file list for Title {title}...')
    blobs = list_title_blobs(title)
    all_blobs[title] = blobs
    print(f'  Found {len(blobs)} XML files')

total_files = sum(len(v) for v in all_blobs.values())
print(f'\nTotal XML files to download: {total_files}')
print('(Already-cached files will be skipped)')
print()

# Download all files
for title, blobs in all_blobs.items():
    already = sum(1 for b in blobs if (Path(DATA_DIR) / b).exists())
    print(f'\nTitle {title}: {len(blobs)} files ({already} already cached)')
    for repo_path in tqdm(blobs, desc=f'  Title {title}', unit='file'):
        download_file(repo_path, Path(DATA_DIR) / repo_path, refresh=False)

# Summary
print('\n✅ Download complete!')
for title in [t.zfill(2) for t in TITLES]:
    count = len(list(Path(DATA_DIR).glob(f'**/{title}/**/*.xml')))
    index = Path(DATA_DIR) / COMAR_BASE / title / 'index.xml'
    status = '✅' if index.exists() else '⚠️'
    print(f'  {status} Title {title}: {count} XML files on Drive')

---
## Step 2: Parse XML into Regulation Dicts

Each COMAR XML file uses `xi:include` to reference sub-files. The parser:
1. Loads the top-level `index.xml` for each title.
2. Recursively resolves all `xi:include` references.
3. Walks the hierarchy: **Title ▶ Subtitle ▶ Chapter ▶ Section (Regulation)**.
4. Returns one Python dict per regulation.

Each dict has fields like `citation`, `chunk_id`, `title_num`, `text`, `effective_date`, `cross_refs`, `chunk_type`.

In [ ]:
# ============================================================
# XML Parser  (fully self-contained, no imports from project)
# ============================================================
import re
from lxml import etree
from pathlib import Path

NS_LIB = 'https://open.law/schemas/library'
NS_XI  = 'http://www.w3.org/2001/XInclude'
_TAG_XI_INCLUDE = f'{{{NS_XI}}}include'

# COMAR pipe-path pattern: 15|01|01|.06
_COMAR_PATH_RE = re.compile(r'([0-9]+)[|]([0-9]+)[|]([0-9]+)[|]([.][0-9]+)')


def _local(el):
    return etree.QName(el.tag).localname if el.tag is not None else ''


def _get_text(el, tag_name):
    for child in el:
        if _local(child) == tag_name:
            return (child.text or '').strip()
    return ''


def _resolve_xincludes(el, base_dir, depth=0):
    if depth > 20:
        return
    for xi in el.findall(f'.//{_TAG_XI_INCLUDE}'):
        href = xi.get('href')
        if not href:
            continue
        path = (Path(base_dir) / href).resolve()
        if not path.exists():
            parent = xi.getparent()
            if parent is not None:
                parent.remove(xi)
            continue
        try:
            included = etree.parse(str(path)).getroot()
            _resolve_xincludes(included, path.parent, depth + 1)
        except Exception:
            parent = xi.getparent()
            if parent is not None:
                parent.remove(xi)
            continue
        parent = xi.getparent()
        if parent is None:
            continue
        idx = list(parent).index(xi)
        xpointer = xi.get('xpointer', '')
        children = list(included) if xpointer else [included]
        parent.remove(xi)
        for offset, child in enumerate(children):
            parent.insert(idx + offset, child)


def _element_text(el):
    parts = []
    def walk(e):
        ln = _local(e)
        if ln in ('annotations', 'annotation'):
            return
        if e.text:
            parts.append(e.text.strip())
        for child in e:
            walk(child)
            if child.tail:
                parts.append(child.tail.strip())
    walk(el)
    return ' '.join(p for p in parts if p)


def _effective_date(section):
    dates = []
    for el in section.iter():
        if _local(el) == 'annotation':
            eff = el.get('effective')
            if eff:
                dates.append(eff)
    return max(dates) if dates else None


def _cross_refs(section):
    refs = []
    seen = set()
    for el in section.iter():
        for attr in ('path', 'href'):
            val = el.get(attr, '')
            m = _COMAR_PATH_RE.search(val)
            if m:
                t, sub, ch, reg = m.groups()
                cit = f'COMAR {t}.{sub}.{ch}.{reg.lstrip(".")}'  # noqa: E501
                if cit not in seen:
                    seen.add(cit)
                    refs.append(cit)
    return refs


def _parse_chapter(chapter_el, title_num, title_name, subtitle_num, subtitle_name):
    chapter_num  = _get_text(chapter_el, 'num').strip().zfill(2)
    chapter_name = _get_text(chapter_el, 'heading')
    regs = []
    for child in chapter_el:
        if _local(child) != 'section':
            continue
        raw_num  = _get_text(child, 'num')
        reg_num  = raw_num.lstrip('.').strip().zfill(2)
        reg_name = _get_text(child, 'heading')
        chunk_id = f'COMAR.{title_num}.{subtitle_num}.{chapter_num}.{reg_num}'
        citation = f'COMAR {title_num}.{subtitle_num}.{chapter_num}.{reg_num}'
        chunk_type = 'definition' if raw_num.lstrip('.') == '01' else 'regulation'
        regs.append({
            'chunk_id'       : chunk_id,
            'citation'       : citation,
            'title_num'      : title_num,
            'subtitle_num'   : subtitle_num,
            'chapter_num'    : chapter_num,
            'regulation_num' : reg_num,
            'title_name'     : title_name,
            'subtitle_name'  : subtitle_name,
            'chapter_name'   : chapter_name,
            'regulation_name': reg_name,
            'text'           : _element_text(child),
            'effective_date' : _effective_date(child),
            'cross_refs'     : _cross_refs(child),
            'chunk_type'     : chunk_type,
        })
    return regs


def _container_prefix(el):
    return _get_text(el, 'prefix').lower()


def parse_comar_xml(xml_path):
    """Parse a COMAR index.xml and return a list of regulation dicts."""
    xml_path = Path(xml_path).resolve()
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    _resolve_xincludes(root, xml_path.parent)
    all_regs = []
    if _container_prefix(root) == 'title':
        title_num  = _get_text(root, 'num').strip().zfill(2)
        title_name = _get_text(root, 'heading')
        for subtitle in root:
            if _local(subtitle) != 'container' or _container_prefix(subtitle) != 'subtitle':
                continue
            subtitle_num  = _get_text(subtitle, 'num').strip().zfill(2)
            subtitle_name = _get_text(subtitle, 'heading')
            for chapter in subtitle:
                if _local(chapter) != 'container' or _container_prefix(chapter) != 'chapter':
                    continue
                all_regs.extend(_parse_chapter(
                    chapter, title_num, title_name, subtitle_num, subtitle_name
                ))
    return all_regs


print('✅ XML parser defined.')

In [ ]:
# ============================================================
# Parse all titles
# ============================================================
import json

all_regulations = []

for title in [t.zfill(2) for t in TITLES]:
    index_path = Path(DATA_DIR) / COMAR_BASE / title / 'index.xml'
    if not index_path.exists():
        print(f'⚠️  index.xml not found for Title {title} (run fetch step first)')
        continue
    print(f'Parsing Title {title}...')
    regs = parse_comar_xml(index_path)
    all_regulations.extend(regs)
    n_def = sum(1 for r in regs if r['chunk_type'] == 'definition')
    subs  = {r['subtitle_num'] for r in regs}
    print(f'  -> {len(regs):,} regulations  |  {n_def} definition sections  |  {len(subs)} subtitles')

print(f'\n✅ Total regulations parsed: {len(all_regulations):,}')

# Save to Drive so you can reload without re-parsing if runtime disconnects
regs_path = f'{GRAPH_DIR}/regulations.json'
with open(regs_path, 'w') as f:
    json.dump(all_regulations, f)
print(f'💾 Regulations saved: {regs_path}')
print(f'   File size: {Path(regs_path).stat().st_size / 1e6:.1f} MB')

**If your Colab runtime restarted**, skip the parse step and just reload from Drive:

In [ ]:
# ============================================================
# OPTIONAL: Reload regulations from Drive (after restart)
# ============================================================
# Run this cell ONLY if your runtime restarted and you want
# to skip re-parsing.  Otherwise, skip it.

import json
from pathlib import Path

regs_path = f'{GRAPH_DIR}/regulations.json'
if Path(regs_path).exists():
    with open(regs_path) as f:
        all_regulations = json.load(f)
    print(f'✅ Loaded {len(all_regulations):,} regulations from {regs_path}')
else:
    print('⚠️  regulations.json not found. Run the parse step first.')

In [ ]:
# ============================================================
# OPTIONAL: Reload chunks from Drive (after restart)
# ============================================================
# Run this cell ONLY if your runtime restarted and you want to
# skip re-parsing + re-chunking.  It restores:
#   - chunks        (list of ~52k dicts, needed for upload)
#   - all_regulations (list of ~3.3k dicts, needed for graph)
# Otherwise skip it.
import json, pickle
from pathlib import Path

chunks_path = f'{GRAPH_DIR}/chunks.pkl'
regs_path   = f'{GRAPH_DIR}/regulations.json'

if Path(chunks_path).exists():
    with open(chunks_path, 'rb') as f:
        chunks = pickle.load(f)
    print(f'✅ Loaded {len(chunks):,} chunks from {chunks_path}')
else:
    print('⚠️  chunks.pkl not found. Run the chunker cell first.')

if Path(regs_path).exists():
    with open(regs_path) as f:
        all_regulations = json.load(f)
    print(f'✅ Loaded {len(all_regulations):,} regulations from {regs_path}')
else:
    print('⚠️  regulations.json not found. Run the parse cell first.')


---
## Step 3: Create Chunks

For each regulation, the chunker creates:
- **Primary chunk** — full regulation text with a breadcrumb prefix (always created).
- **Subsection chunks** — if the primary chunk exceeds 600 tokens, it is split at
  paragraph markers (`A.`, `B.`, `(1)`, `(a)` …) into smaller child chunks.

Token counting uses the `cl100k_base` tokenizer (same as GPT-4).

In [ ]:
# ============================================================
# Chunker  (fully self-contained)
# ============================================================
import re, tiktoken

SUBSECTION_TOKEN_THRESHOLD = 600
_TOKENIZER = tiktoken.get_encoding('cl100k_base')

# Split at A. B. (1) (a) etc. at word boundaries
# Uses character classes to avoid backslash escaping issues
_SUBSEC_RE = re.compile(
    r'(?:^|(?<= ))'
    r'(?=[A-Z][.] |[(][0-9]+[)] |[(][a-z]+[)] )'
)

# Extract "term" means ... definitions
_DEFN_PAT = re.compile(
    r'["\u201c]([^"\u201d]{1,80})["\u201d]'
    r'[ ]+(?:means|has the meaning|refers to|is defined as)'
    r'([^"\u201d]{5,500})',
    re.IGNORECASE,
)


def _tok(text):
    return len(_TOKENIZER.encode(text))


def _breadcrumb(reg):
    return (
        f"Title {reg['title_num']} \u2014 {reg['title_name']} > "
        f"Subtitle {reg['subtitle_num']} \u2014 {reg['subtitle_name']} > "
        f"Chapter {reg['chapter_num']} \u2014 {reg['chapter_name']} > "
        f"Regulation .{reg['regulation_num']} {reg['regulation_name']}"
    )


def create_chunks(regulations):
    all_chunks = []
    definitions_lookup = {}

    for reg in regulations:
        bc = _breadcrumb(reg)
        chunk_text = f'{bc}\n\n{reg["text"]}'
        tc = _tok(chunk_text)

        # Primary chunk
        primary = {k: v for k, v in reg.items() if k != 'text'}
        primary.update({
            'chunk_text' : chunk_text,
            'parent_id'  : None,
            'word_count' : len(chunk_text.split()),
            'token_count': tc,
        })
        all_chunks.append(primary)

        # Subsection chunks (only for long regulations)
        if tc > SUBSECTION_TOKEN_THRESHOLD:
            segments = [s.strip() for s in _SUBSEC_RE.split(reg['text']) if s.strip()]
            if len(segments) > 1:
                for idx, seg in enumerate(segments):
                    sub_text = f'{bc}\n\n{seg}'
                    sub = {k: v for k, v in reg.items() if k != 'text'}
                    sub.update({
                        'chunk_id'   : f"{reg['chunk_id']}.sub.{idx}",
                        'chunk_text' : sub_text,
                        'chunk_type' : 'subsection',
                        'parent_id'  : reg['chunk_id'],
                        'word_count' : len(sub_text.split()),
                        'token_count': _tok(sub_text),
                    })
                    all_chunks.append(sub)

        # Extract definitions from .01 Definition sections
        if reg.get('chunk_type') == 'definition':
            for m in _DEFN_PAT.finditer(reg['text']):
                term = m.group(1).strip()
                defn = m.group(2).strip()[:400]
                if term and defn:
                    definitions_lookup[term.lower()] = {
                        'definition_text'  : f'"{term}" means {defn}',
                        'source_citation'  : reg['citation'],
                    }

    return all_chunks, definitions_lookup


print('✅ Chunker defined.')

In [ ]:
# ============================================================
# Run chunker
# ============================================================
import json, pickle
from pathlib import Path

print('Creating chunks...')
chunks, definitions = create_chunks(all_regulations)

n_primary    = len(all_regulations)
n_subsection = sum(1 for c in chunks if c.get('chunk_type') == 'subsection')
n_total      = len(chunks)

print(f'\n📊 Chunking results:')
print(f'   Primary chunks     : {n_primary:,}')
print(f'   Subsection chunks  : {n_subsection:,}')
print(f'   Total chunks       : {n_total:,}')
print(f'   Definitions found  : {len(definitions):,}')

# Show token size distribution
toks = [c['token_count'] for c in chunks]
print(f'\n   Token counts (chunks):')
print(f'   min={min(toks)}, median={sorted(toks)[len(toks)//2]}, max={max(toks)}')

# Save definitions
defs_path = f'{GRAPH_DIR}/definitions.json'
with open(defs_path, 'w') as f:
    json.dump(definitions, f, indent=2)
print(f'\n💾 Definitions saved: {defs_path}')

# Save chunks to Drive so they survive a runtime restart
# (pickle is ~5x faster to load than JSON for a list this large)
chunks_path = f'{GRAPH_DIR}/chunks.pkl'
with open(chunks_path, 'wb') as f:
    pickle.dump(chunks, f, protocol=pickle.HIGHEST_PROTOCOL)
size_mb = Path(chunks_path).stat().st_size / 1e6
print(f'💾 Chunks saved     : {chunks_path}  ({size_mb:.0f} MB)')
print('\n✅ All data saved to Drive — safe to restart runtime.')


---
## Step 4: Build Knowledge Graph

The knowledge graph is a [NetworkX](https://networkx.org/) directed graph with:

| Node type | Example ID | Description |
|-----------|-----------|-------------|
| `title` | `TITLE.15` | Title container |
| `subtitle` | `SUBTITLE.15.06` | Subtitle container |
| `chapter` | `CHAPTER.15.06.02` | Chapter container |
| `regulation` | `COMAR.15.06.02.01` | Individual regulation |

Edge types: `CONTAINS` (hierarchy), `REFERENCES` (cross-refs), `DEFINES` (definition → chapter).

The graph is saved to Google Drive as a pickle file and used by the local RAG pipeline for graph-based context expansion.

In [ ]:
# ============================================================
# Build Knowledge Graph
# ============================================================
import networkx as nx
import pickle
from pathlib import Path


def build_knowledge_graph(regulations, save_path=None):
    """Build a directed knowledge graph from COMAR regulations."""
    graph = nx.DiGraph()
    cit_to_id = {r['citation']: r['chunk_id'] for r in regulations}

    for reg in regulations:
        t, sub, ch = reg['title_num'], reg['subtitle_num'], reg['chapter_num']
        reg_id = reg['chunk_id']
        tid = f'TITLE.{t}'
        sid = f'SUBTITLE.{t}.{sub}'
        cid = f'CHAPTER.{t}.{sub}.{ch}'

        if not graph.has_node(tid):
            graph.add_node(tid, node_type='title', title_num=t,
                           title_name=reg.get('title_name', ''))
        if not graph.has_node(sid):
            graph.add_node(sid, node_type='subtitle', subtitle_num=sub,
                           subtitle_name=reg.get('subtitle_name', ''))
            graph.add_edge(tid, sid, edge_type='CONTAINS')
        if not graph.has_node(cid):
            graph.add_node(cid, node_type='chapter', chapter_num=ch,
                           chapter_name=reg.get('chapter_name', ''))
            graph.add_edge(sid, cid, edge_type='CONTAINS')

        graph.add_node(reg_id, node_type='regulation',
                       **{k: v for k, v in reg.items() if k != 'text'})
        graph.add_edge(cid, reg_id, edge_type='CONTAINS')

        if reg.get('chunk_type') == 'definition':
            graph.add_edge(reg_id, cid, edge_type='DEFINES')

    # REFERENCES edges (second pass ensures all nodes exist)
    for reg in regulations:
        for cited in reg.get('cross_refs', []):
            tgt = cit_to_id.get(cited)
            if tgt and tgt != reg['chunk_id']:
                graph.add_edge(reg['chunk_id'], tgt, edge_type='REFERENCES')

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, 'wb') as f:
            pickle.dump(graph, f, protocol=pickle.HIGHEST_PROTOCOL)
        size_kb = Path(save_path).stat().st_size / 1024
        print(f'💾 Graph saved: {save_path}  ({size_kb:.0f} KB)')

    return graph


print('Building knowledge graph...')
graph_path = f'{GRAPH_DIR}/comar_graph.pkl'
graph = build_knowledge_graph(all_regulations, save_path=graph_path)

contains = sum(1 for _, _, d in graph.edges(data=True) if d.get('edge_type') == 'CONTAINS')
refs     = sum(1 for _, _, d in graph.edges(data=True) if d.get('edge_type') == 'REFERENCES')
defines  = sum(1 for _, _, d in graph.edges(data=True) if d.get('edge_type') == 'DEFINES')

print(f'\n📊 Knowledge graph summary:')
print(f'   Nodes      : {graph.number_of_nodes():,}')
print(f'   Edges      : {graph.number_of_edges():,}')
print(f'     CONTAINS   : {contains:,}')
print(f'     REFERENCES : {refs:,}')
print(f'     DEFINES    : {defines:,}')

---
## Step 5: Qdrant Cloud — Setup Guide

If you haven't already set up Qdrant Cloud, follow these steps:

### 5a. Create a Qdrant Cloud account
1. Go to **https://cloud.qdrant.io** and sign up (free).
2. Click **Create Cluster**.
3. Choose any cloud provider and region (e.g. GCP us-east4).
4. Select the **Free tier** (1 GB, 0 cost).
5. Click **Create** and wait ~30 seconds.

### 5b. Get your credentials
After the cluster is ready:
- **Endpoint URL**: shown on the cluster dashboard, e.g. `https://abc-123.us-east4-0.gcp.cloud.qdrant.io`
- **API Key**: click **API Keys** tab → **Create API Key** → copy the key.

Paste both into the **Configuration cell** at the top of this notebook.

> **Note**: The Free tier gives you 1 GB of storage. The COMAR collection (52,528 vectors × 1024 dims) uses about 200–400 MB, well within the limit.

In [ ]:
# ============================================================
# Step 5c: Create Qdrant Cloud Collection
# ============================================================
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm

# Connect to Qdrant Cloud
print(f'Connecting to Qdrant Cloud at {QDRANT_URL}...')
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=30)

# Test connection
collections = client.get_collections().collections
existing_names = {c.name for c in collections}
print(f'✅ Connected! Existing collections: {list(existing_names) or "(none)"}')

DENSE_DIM = 1024

if COLLECTION_NAME in existing_names:
    info = client.get_collection(COLLECTION_NAME)
    print(f'\n⚠️  Collection "{COLLECTION_NAME}" already exists: {info.points_count} points')
    print('   Skipping creation. To start fresh, run:')
    print(f'   client.delete_collection("{COLLECTION_NAME}")')
else:
    print(f'\nCreating collection "{COLLECTION_NAME}"...')
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            'dense': qm.VectorParams(
                size=DENSE_DIM,
                distance=qm.Distance.COSINE,
                on_disk=False,
            ),
        },
        sparse_vectors_config={
            'sparse': qm.SparseVectorParams(
                index=qm.SparseIndexParams(on_disk=False)
            ),
        },
        optimizers_config=qm.OptimizersConfigDiff(indexing_threshold=20_000),
    )

    # Payload indexes for fast filtering
    for field, schema in [
        ('title_num',      qm.PayloadSchemaType.KEYWORD),
        ('subtitle_num',   qm.PayloadSchemaType.KEYWORD),
        ('chunk_type',     qm.PayloadSchemaType.KEYWORD),
        ('effective_date', qm.PayloadSchemaType.KEYWORD),
        ('chunk_id',       qm.PayloadSchemaType.KEYWORD),
    ]:
        client.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field,
            field_schema=schema,
        )

    print(f'\n✅ Collection "{COLLECTION_NAME}" created successfully.')
    print('   Vectors  : dense (1024-dim, Cosine) + sparse (BM25-like)')
    print('   Indexes  : title_num, subtitle_num, chunk_type, effective_date, chunk_id')

---
## Step 6: Load BGE-M3 Embedding Model

**BAAI/bge-m3** produces three vector types per text:
- **Dense** (1024-dim) — semantic similarity (dot product / cosine)
- **Sparse** (token weights) — BM25-equivalent lexical matching
- **ColBERT** (token-level) — late interaction (we skip this to save memory)

The model weights are ~2.3 GB and will be downloaded from Hugging Face on the first run.
Subsequent runs reuse the Colab cache (within the same session).

> **FP16 note**: On T4 GPU we use 16-bit precision (`use_fp16=True`), which halves
> memory usage and roughly doubles throughput.

> **If you see an ImportError about `is_torch_fx_available`**, run the cell below
> first — it downgrades FlagEmbedding to the last version compatible with Colab's
> `transformers>=4.44`.

In [ ]:
# ============================================================
# Step 6: Load BGE-M3 on GPU
# ============================================================

# ── Compatibility fix (two layers) ───────────────────────────
# FlagEmbedding 1.3.x imports `is_torch_fx_available` from
# transformers, but that symbol was removed in transformers>=4.44
# (Colab's current default).  We fix this two ways:
#
# Layer 1 — patch the live module object so `from X import Y` works
import transformers.utils.import_utils as _tui
if not hasattr(_tui, 'is_torch_fx_available'):
    _tui.is_torch_fx_available = lambda: False
    print('Layer 1 patch applied ✅')

# Layer 2 — patch the source file + clear bytecode cache so even
# a previously-failed (partial) import can be retried cleanly
import pathlib, glob as _g
for _f in _g.glob('/usr/local/lib/python3*/dist-packages/FlagEmbedding/'
                  'inference/reranker/decoder_only/models/modeling_minicpm_reranker.py'):
    _p = pathlib.Path(_f)
    _bad = 'from transformers.utils.import_utils import is_torch_fx_available'
    _src = _p.read_text()
    if _bad in _src:
        _p.write_text(_src.replace(_bad, 'is_torch_fx_available = lambda: False  # compat patch'))
        for _c in (_p.parent / '__pycache__').glob('modeling_minicpm_reranker*.pyc'):
            _c.unlink(missing_ok=True)
        print(f'Layer 2 patch applied ({_p.name}) ✅')

# Clear any failed partial import that may be stuck in sys.modules
import sys as _sys
_removed = [k for k in list(_sys.modules) if 'minicpm_reranker' in k]
for k in _removed:
    del _sys.modules[k]
if _removed:
    print(f'Cleared {len(_removed)} stale module cache entries ✅')

# ── Load model ───────────────────────────────────────────────
import torch
from FlagEmbedding import BGEM3FlagModel

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = device == 'cuda'

print(f'\nDevice : {device} | FP16 : {use_fp16}')
print('Loading BAAI/bge-m3 (~2.3 GB on first run)...')

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=use_fp16, device=device)

print(f'\n✅ BGE-M3 loaded on {device}')
if device == 'cuda':
    print(f'   GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')


---
## Step 7: Upload Chunks to Qdrant Cloud

This is the longest step (~2–3 hours on T4 GPU for 52,528 chunks).

**Resume support**: After each batch is uploaded, the batch index is saved to
`COMAR_RAG/upload_checkpoint.txt` on your Drive. If your Colab session
disconnects, just re-run this cell — it will automatically pick up where it left off.

**Point IDs**: Each chunk gets a deterministic `uuid5` ID based on its `chunk_id` string.
This means re-running the upload is safe (upsert, not insert).

In [ ]:
# ============================================================
# Step 7: Embed and Upload with Resume Support
# ============================================================
import uuid, time, json, pickle
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from tqdm import tqdm

# ── Guard: define any variables not set by earlier cells ─────
# (Needed when this cell is run after a runtime restart without
#  re-running all earlier cells in order.)
if 'GRAPH_DIR' not in dir():
    GRAPH_DIR = '/content/drive/MyDrive/COMAR_RAG/data'
if 'CKPT_FILE' not in dir():
    CKPT_FILE = '/content/drive/MyDrive/COMAR_RAG/upload_checkpoint.txt'
if 'EMBED_BATCH_SIZE' not in dir():
    EMBED_BATCH_SIZE = 16
if 'COLLECTION_NAME' not in dir():
    COLLECTION_NAME = 'comar_regulations'
if 'QDRANT_URL' not in dir() or not QDRANT_URL:
    raise RuntimeError('QDRANT_URL is not set. Run the Configuration cell (Step 3).')
if 'QDRANT_API_KEY' not in dir() or not QDRANT_API_KEY:
    raise RuntimeError('QDRANT_API_KEY is not set. Run the Configuration cell (Step 3).')

# ── Guard: load chunks from Drive if not in memory ───────────
if 'chunks' not in dir():
    chunks_path = f'{GRAPH_DIR}/chunks.pkl'
    if not Path(chunks_path).exists():
        raise RuntimeError(
            'chunks.pkl not found on Drive.\n'
            'Run: Mount Drive → Config → Reload chunks cell → Model cell → this cell.'
        )
    print(f'Loading chunks from Drive...')
    with open(chunks_path, 'rb') as f:
        chunks = pickle.load(f)
    print(f'  Loaded {len(chunks):,} chunks ✅')

# ── Guard: reconnect Qdrant client if not in memory ──────────
if 'client' not in dir():
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=30)
    print(f'Reconnected Qdrant client ✅')

# ── Guard: model must be loaded — no auto-load (too slow) ────
if 'model' not in dir():
    raise RuntimeError('BGE-M3 model not loaded. Run the Step 6 (Load BGE-M3) cell first.')

# ─────────────────────────────────────────────────────────────
_UUID_NS = uuid.UUID('6ba7b810-9dad-11d1-80b4-00c04fd430c8')


def _chunk_to_uuid(chunk_id):
    return str(uuid.uuid5(_UUID_NS, chunk_id))


def _load_checkpoint():
    p = Path(CKPT_FILE)
    if p.exists():
        try:
            return int(p.read_text().strip())
        except ValueError:
            pass
    return 0


def _save_checkpoint(batch_idx):
    Path(CKPT_FILE).write_text(str(batch_idx))


# --- Prepare batches ---
start_batch  = _load_checkpoint()
total_chunks = len(chunks)
batches      = [chunks[i : i + EMBED_BATCH_SIZE] for i in range(0, total_chunks, EMBED_BATCH_SIZE)]
n_batches    = len(batches)

if start_batch > 0:
    already_done = start_batch * EMBED_BATCH_SIZE
    print(f'🔄 Resuming from batch {start_batch}/{n_batches}')
    print(f'   ({already_done:,} of {total_chunks:,} chunks already uploaded)')
else:
    print(f'Starting upload: {total_chunks:,} chunks → {n_batches} batches of {EMBED_BATCH_SIZE}')

print(f'Target: {QDRANT_URL} / {COLLECTION_NAME}')
print()

t_start = time.time()

for batch_idx in tqdm(range(start_batch, n_batches), desc='Uploading', unit='batch'):
    batch = batches[batch_idx]
    texts = [c['chunk_text'] for c in batch]

    # Embed: dense + sparse only (skip ColBERT to save VRAM)
    raw = model.encode(
        texts,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )

    dense_vecs = [
        v.tolist() if hasattr(v, 'tolist') else list(v)
        for v in raw['dense_vecs']
    ]
    sparse_vecs = [
        {int(k): float(v) for k, v in sv.items()}
        for sv in raw['lexical_weights']
    ]

    # Build Qdrant points
    points = []
    for chunk, dvec, svec in zip(batch, dense_vecs, sparse_vecs):
        point_id = _chunk_to_uuid(chunk['chunk_id'])
        payload  = dict(chunk)
        sp_idx   = list(svec.keys())
        sp_val   = [svec[i] for i in sp_idx]
        points.append(qm.PointStruct(
            id=point_id,
            vector={
                'dense' : dvec,
                'sparse': qm.SparseVector(indices=sp_idx, values=sp_val),
            },
            payload=payload,
        ))

    # Upsert to Qdrant Cloud
    client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    _save_checkpoint(batch_idx + 1)

elapsed = time.time() - t_start
print(f'\n✅ Upload complete!')
print(f'   {total_chunks:,} chunks in {elapsed/60:.1f} minutes')

if Path(CKPT_FILE).exists():
    Path(CKPT_FILE).unlink()
    print('✅ Checkpoint cleared.')


---
## Step 8: Verify Collection and Test Search

These cells confirm that:
1. The expected number of points is in the collection.
2. Hybrid search (dense + sparse → RRF fusion) returns relevant results.

In [ ]:
# ============================================================
# Step 8a: Verify collection statistics
# ============================================================
info = client.get_collection(COLLECTION_NAME)

print(f'✅ Collection: {COLLECTION_NAME}')
print(f'   Points  : {info.points_count:,}')
print(f'   Status  : {info.status}')
if info.payload_schema:
    print(f'   Indexed : {list(info.payload_schema.keys())}')
# vectors_count removed in newer qdrant-client; use points_count instead
vc = getattr(info, 'vectors_count', None)
if vc is not None:
    print(f'   Vectors : {vc:,}')

# Quick sample: show first 5 records
pts, _ = client.scroll(
    collection_name=COLLECTION_NAME, limit=5, with_payload=True, with_vectors=False
)
print(f'\n📋 Sample records:')
for pt in pts:
    cit  = pt.payload.get('citation', 'N/A')
    ctyp = pt.payload.get('chunk_type', '?')
    toks = pt.payload.get('token_count', '?')
    print(f'   {cit:30s}  type={ctyp:12s}  tokens={toks}')


In [ ]:
# ============================================================
# Step 8b: Run a test hybrid search query
# ============================================================
from qdrant_client.http.models import Prefetch, FusionQuery

TEST_QUERY = 'What is the definition of a pesticide dealer under Title 15?'
print(f'Test query: "{TEST_QUERY}"\n')

# Embed the query with BGE-M3
q_raw = model.encode([TEST_QUERY], return_dense=True, return_sparse=True, return_colbert_vecs=False)
q_dense  = q_raw['dense_vecs'][0].tolist()
q_sparse_raw = q_raw['lexical_weights'][0]
q_sparse = qm.SparseVector(
    indices=list(q_sparse_raw.keys()),
    values=[float(v) for v in q_sparse_raw.values()],
)

# Hybrid search: dense + sparse fused with RRF
results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=q_dense,  using='dense',  limit=10),
        Prefetch(query=q_sparse, using='sparse', limit=10),
    ],
    query=FusionQuery(fusion=qm.Fusion.RRF),
    limit=5,
    with_payload=True,
)

print(f'Top-5 results:')
for i, pt in enumerate(results.points):
    cit  = pt.payload.get('citation', 'N/A')
    text = pt.payload.get('chunk_text', '')[:150].replace('\n', ' ')
    print(f'\n[{i+1}] {cit}  (score={pt.score:.4f})')
    print(f'    {text}...')

---
## Step 9: Export Snapshot from Qdrant Cloud

A **snapshot** is a portable binary backup of the entire Qdrant collection (vectors + payloads).
We create a snapshot on Qdrant Cloud and download it to Google Drive.
You then transfer it to your local machine and restore it to your local Docker Qdrant instance.

**Expected snapshot size**: ~200–500 MB for the COMAR collection.

In [ ]:
# ============================================================
# Step 9a: Create snapshot on Qdrant Cloud
# ============================================================
import requests as req
from pathlib import Path

_headers = {'api-key': QDRANT_API_KEY, 'Content-Type': 'application/json'}
_base    = QDRANT_URL.rstrip('/')

print(f'Creating snapshot of "{COLLECTION_NAME}" on Qdrant Cloud...')
r = req.post(
    f'{_base}/collections/{COLLECTION_NAME}/snapshots',
    headers=_headers,
    timeout=300,
)
r.raise_for_status()
snap_info = r.json()
snap_name = snap_info['result']['name']
print(f'✅ Snapshot created: {snap_name}')

# Download snapshot to Drive
print(f'\nDownloading snapshot to Drive (may take a few minutes)...')
snap_url  = f'{_base}/collections/{COLLECTION_NAME}/snapshots/{snap_name}'
snap_path = Path(GRAPH_DIR) / snap_name

r2 = req.get(snap_url, headers=_headers, stream=True, timeout=600)
r2.raise_for_status()

total_size = int(r2.headers.get('content-length', 0))
downloaded = 0
with open(snap_path, 'wb') as fh:
    for chunk_data in r2.iter_content(chunk_size=1024 * 1024):
        fh.write(chunk_data)
        downloaded += len(chunk_data)
        if total_size:
            pct = downloaded / total_size * 100
            print(f'\r   {pct:.1f}%  ({downloaded/1e6:.0f} / {total_size/1e6:.0f} MB)', end='')

size_mb = snap_path.stat().st_size / 1e6
print(f'\n\n✅ Snapshot saved to Google Drive:')
print(f'   {snap_path}')
print(f'   Size: {size_mb:.1f} MB')
print(f'\nSnap name (needed for local restore): {snap_name}')

---
## Step 10: Transfer to Local Docker Qdrant

Now run these steps **on your local machine** (not in Colab).

### Option A: Restore directly from Qdrant Cloud URL (easiest)

This tells your local Qdrant to download the snapshot directly from Qdrant Cloud over the internet.
No manual file download required.

```python
from qdrant_client import QdrantClient

local = QdrantClient(host='localhost', port=6333)

QDRANT_CLOUD_URL = 'https://abc-123.us-east4-0.gcp.cloud.qdrant.io'  # your Cloud URL
QDRANT_API_KEY   = 'your-api-key'
SNAP_NAME        = 'comar_regulations-XXX-YYY.snapshot'               # from Step 9

snapshot_url = f'{QDRANT_CLOUD_URL}/collections/comar_regulations/snapshots/{SNAP_NAME}'

local.recover_snapshot(
    collection_name='comar_regulations',
    location=snapshot_url,
    api_key=QDRANT_API_KEY,    # authenticates the download from Qdrant Cloud
)
print('Done!')
```

### Option B: Manual file transfer (if Option A fails)

1. **Download** the `.snapshot` file from Google Drive to your Mac.
2. Make sure local Qdrant is running:
   ```bash
   cd "COMAR RAG"
   docker compose up -d
   ```
3. Upload the snapshot via the REST API:
   ```bash
   curl -X POST http://localhost:6333/collections/comar_regulations/snapshots/upload \
     -H 'Content-Type: multipart/form-data' \
     -F 'snapshot=@/path/to/comar_regulations-XXX-YYY.snapshot'
   ```
4. Verify:
   ```bash
   curl http://localhost:6333/collections/comar_regulations | python3 -m json.tool
   ```

### Verify (run locally)

```python
from qdrant_client import QdrantClient
c = QdrantClient(host='localhost', port=6333)
info = c.get_collection('comar_regulations')
print(f'Points : {info.points_count}')   # should be ~52,528
print(f'Status : {info.status}')         # should be green
```

Once the local collection is verified, your COMAR RAG API (FastAPI on port 8000) will
automatically detect the populated collection via `settings.qdrant_ready` and switch
from stub mode to full hybrid search.

In [ ]:
# ============================================================
# Step 10 helper: run this cell LOCALLY (not in Colab)
# to restore the snapshot from Qdrant Cloud to local Docker.
# ============================================================
#
# Copy this cell's code to a local Python script or terminal.
# Make sure your Docker Qdrant is running first:
#   docker compose up -d   (from inside COMAR RAG directory)

import sys

# Fill in these values from Step 9 output:
CLOUD_URL  = ''   # your Qdrant Cloud endpoint
CLOUD_KEY  = ''   # your Qdrant Cloud API key
SNAP_NAME  = ''   # e.g. 'comar_regulations-2024-01-01-12-00-00.snapshot'

if not all([CLOUD_URL, CLOUD_KEY, SNAP_NAME]):
    print('Fill in CLOUD_URL, CLOUD_KEY, and SNAP_NAME above, then run this cell locally.')
    sys.exit(0)

from qdrant_client import QdrantClient

local_client = QdrantClient(host='localhost', port=6333)
snapshot_url = f'{CLOUD_URL.rstrip("/")}/collections/comar_regulations/snapshots/{SNAP_NAME}'

print(f'Restoring collection from:\n  {snapshot_url}')
print('This may take several minutes...')

local_client.recover_snapshot(
    collection_name='comar_regulations',
    location=snapshot_url,
    api_key=CLOUD_KEY,
)

info = local_client.get_collection('comar_regulations')
print(f'\n✅ Restore complete!')
print(f'   Points : {info.points_count:,}')
print(f'   Status : {info.status}')

---
## What Next?

With the Qdrant collection populated on your local machine:

1. **Start the FastAPI server** (from your local COMAR RAG directory):
   ```bash
   PYTHONPATH=. uvicorn api.main:app --port 8000 --reload
   ```

2. **Start the React frontend**:
   ```bash
   cd frontend && npm run dev
   ```

3. **Check the health endpoint**:
   ```bash
   curl http://localhost:8000/api/health
   # Should return: {"qdrant_ready": true, "llm_ready": true}
   ```

4. **Copy your Qdrant collection to the graph file** (it was saved to Drive already):
   - Copy `data/comar_graph.pkl` from Drive to your local `COMAR RAG/data/` directory.
   - The LangGraph pipeline uses this for graph-based context expansion.

---
*Built for COMAR RAG — Maryland COMAR Titles 15 & 26 Research Project*